In [4]:
import os
import numpy as np
import pandas as pd
import folium
import branca
from folium.plugins import MarkerCluster
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize
import seaborn as sns
from datetime import datetime
import math
import glob

# Configuration Parameters
RUNTIME_CONFIG = {
    'DATA_FILES_PATTERN': 'JC2024*citibiketripdata.csv', 
    'DATA_DIR': 'data/citibike', 
    'OUTPUT_DIR': 'results', 
    'MODE': 'visualize', 
    'MAX_STATIONS': 50, 
    'SAMPLE_RATE': 1, 
    'DISTANCE_THRESHOLD': 2.0, 
}

# Citibike time period configurations
TIME_PERIODS = {
    "Morning Peak (7-10 AM)": (7, 10),
    "Afternoon (10 AM-4 PM)": (10, 16),
    "Evening Peak (4-7 PM)": (16, 19),
    "All Day": (0, 24)
}

def haversine_distance(lat1, lon1, lat2, lon2):
    """Calculate haversine distance between two points in kilometers"""
    # Convert decimal degrees to radians
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    
    # Haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    c = 2 * math.asin(math.sqrt(a))
    r = 6371 # Radius of earth in kilometers
    
    return c * r

def create_bezier_curve_with_strength(lon1, lat1, lon2, lat2, curve_strength=0.2, num_points=20):
    """Create smooth bezier curve between two points with customizable curve strength"""
    # Calculate midpoint
    mid_lon = (lon1 + lon2) / 2
    mid_lat = (lat1 + lat2) / 2
    
    # Calculate control point perpendicular to the line
    dx = lon2 - lon1
    dy = lat2 - lat1
    
    # Calculate control point perpendicular to the line
    control_lon = mid_lon - dy * curve_strength
    control_lat = mid_lat + dx * curve_strength
    
    # Generate curve points
    t = np.linspace(0, 1, num_points)
    curve = []
    for ti in t:
        x = (1-ti)**2 * lon1 + 2*(1-ti)*ti * control_lon + ti**2 * lon2
        y = (1-ti)**2 * lat1 + 2*(1-ti)*ti * control_lat + ti**2 * lat2
        curve.append([y, x])  # Note: Folium expects [lat, lon] not [lon, lat]
    
    return curve

def create_bezier_curve(lon1, lat1, lon2, lat2, num_points=20):
    """Create smooth bezier curve between two points with dynamic strength"""
    # Calculate distance for dynamic curve strength
    dx = lon2 - lon1
    dy = lat2 - lat1
    distance = math.sqrt(dx**2 + dy**2)
    
    # Dynamic curve strength based on distance
    curve_strength = min(0.3, 0.05 + 0.1 * (distance / 0.05))
    
    return create_bezier_curve_with_strength(lon1, lat1, lon2, lat2, curve_strength, num_points)

def load_citibike_data(data_dir, data_files_pattern, sample_rate=1.0):
    """Load and preprocess Citibike data from multiple files"""
    print(f"Loading Citibike data with {sample_rate*100:.1f}% sampling rate")
    
    # Find all matching files
    file_pattern = os.path.join(data_dir, data_files_pattern)
    file_list = glob.glob(file_pattern)
    
    if not file_list:
        raise FileNotFoundError(f"No data files found matching pattern: {file_pattern}")
    
    file_list.sort() # Ensure files are processed in order
    print(f"Found {len(file_list)} files to process")
    
    # Load data from each file
    all_trips = []
    
    for file_path in file_list:
        file_name = os.path.basename(file_path)
        print(f"Processing file: {file_name}")
        
        # Extract month from filename (assuming format JC2024MMcitibiketripdata.csv)
        try:
            month = int(file_name[6:8])
            print(f"Extracted month: {month}")
        except (ValueError, IndexError):
            month = 0
            print(f"Could not extract month from filename: {file_name}, using placeholder")
        
        # Load data in chunks to save memory
        chunks = []
        
        for chunk in pd.read_csv(file_path, chunksize=10000):
            # Sample the chunk to reduce memory usage
            sampled_chunk = chunk.sample(frac=sample_rate, random_state=42)
            chunks.append(sampled_chunk)
        
        # Combine chunks for this month
        if chunks:
            monthly_trips = pd.concat(chunks)
            print(f"Loaded {len(monthly_trips)} sampled trips from {file_name}")
            
            # Add month column if not already present
            if 'month' not in monthly_trips.columns:
                monthly_trips['month'] = month
            
            all_trips.append(monthly_trips)
        else:
            print(f"No data loaded from {file_name}")
    
    # Combine all monthly data
    if not all_trips:
        raise ValueError("No trip data was loaded from any file")
    
    trips_df = pd.concat(all_trips)
    print(f"Combined dataset contains {len(trips_df)} trips")
    
    # Basic data cleaning
    trips_df = trips_df.dropna(subset=['start_station_name', 'end_station_name', 
                                      'start_lat', 'start_lng', 
                                      'end_lat', 'end_lng'])
    
    # Convert timestamps to datetime
    print("Converting timestamps to datetime...")
    trips_df['started_at'] = pd.to_datetime(trips_df['started_at'], errors='coerce')
    trips_df['ended_at'] = pd.to_datetime(trips_df['ended_at'], errors='coerce')
    
    # Extract time features
    trips_df['start_hour'] = trips_df['started_at'].dt.hour
    trips_df['start_day'] = trips_df['started_at'].dt.dayofweek # Monday=0, Sunday=6
    trips_df['is_weekend'] = trips_df['start_day'].apply(lambda x: 1 if x >= 5 else 0)
    
    # Make sure month is available
    if 'month' not in trips_df.columns:
        trips_df['month'] = trips_df['started_at'].dt.month
    
    # Calculate trip duration in minutes
    trips_df['duration_minutes'] = (trips_df['ended_at'] - trips_df['started_at']).dt.total_seconds() / 60
    
    # Filter out unrealistic trips
    trips_df = trips_df[(trips_df['duration_minutes'] > 0) & 
                        (trips_df['duration_minutes'] < 24*60)] # Less than 24 hours
    
    print(f"After cleaning: {len(trips_df)} trips")
    
    return trips_df

def get_top_stations(trips_df, max_stations):
    """Get the top stations by trip count and create station mapping"""
    # Get unique stations
    start_counts = trips_df['start_station_name'].value_counts()
    end_counts = trips_df['end_station_name'].value_counts()
    combined_counts = start_counts.add(end_counts, fill_value=0)
    
    # Take the top stations
    max_stations = min(max_stations, len(combined_counts))
    top_stations = combined_counts.sort_values(ascending=False).head(max_stations).index.tolist()
    
    print(f"Using top {len(top_stations)} stations")
    
    # Create station mapping
    station_mapping = {station: idx for idx, station in enumerate(top_stations)}
    
    # Create station data dictionary
    station_data = {}
    
    for station in top_stations:
        # Get station coordinates from the first occurrence
        start_rows = trips_df[trips_df['start_station_name'] == station]
        end_rows = trips_df[trips_df['end_station_name'] == station]
        
        if not start_rows.empty:
            lat = start_rows['start_lat'].iloc[0]
            lng = start_rows['start_lng'].iloc[0]
        elif not end_rows.empty:
            lat = end_rows['end_lat'].iloc[0]
            lng = end_rows['end_lng'].iloc[0]
        else:
            lat, lng = 0, 0
            print(f"No coordinate data found for station: {station}")
        
        # Calculate station popularity
        popularity = (
            trips_df['start_station_name'].value_counts().get(station, 0) +
            trips_df['end_station_name'].value_counts().get(station, 0)
        )
        
        station_data[station_mapping[station]] = {
            'name': station,
            'lat': lat,
            'lng': lng,
            'popularity': popularity
        }
    
    # Create a dataframe from station_data for easy access
    stations_df = pd.DataFrame([
        {
            'name': data['name'],
            'lat': data['lat'],
            'lng': data['lng'],
            'popularity': data['popularity']
        }
        for idx, data in station_data.items()
    ])
    
    return station_mapping, station_data, top_stations, stations_df

def create_interactive_station_map(stations_df, map_title="Bike Stations", output_dir="results"):
    """Create an interactive map of bike stations with multiple layer options and controls"""
    # Calculate map center
    center_lat = stations_df['lat'].mean()
    center_lng = stations_df['lng'].mean()
    
    # Create base map with multiple tile options
    m = folium.Map(
        location=[center_lat, center_lng],
        zoom_start=13,
        tiles=None,  # No default tiles, we'll add them as layers
        control_scale=True
    )
    
    # Add multiple tile layers with proper attribution
    folium.TileLayer('cartodbpositron', name='CartoDB Positron', control=True).add_to(m)
    folium.TileLayer('openstreetmap', name='OpenStreetMap', control=True).add_to(m)
    folium.TileLayer('cartodbdark_matter', name='CartoDB Dark Matter', control=True).add_to(m)
    
    # Add Esri tile layers with attribution
    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
        attr="Esri",
        name="Esri Satellite",
        control=True
    ).add_to(m)
    
    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
        attr="Esri",
        name="Esri Standard",
        control=True
    ).add_to(m)
    
    # Create feature groups for better organization
    stations_group = folium.FeatureGroup(name="All Stations", show=True)
    popular_group = folium.FeatureGroup(name="Popular Stations", show=False)
    clusters_group = MarkerCluster(name="Clustered Stations")
    
    # Find threshold for popular stations (top 25%)
    popularity_threshold = stations_df['popularity'].quantile(0.75)
    
    # Create a colormap for station markers based on popularity
    max_popularity = stations_df['popularity'].max()
    colormap = branca.colormap.LinearColormap(
        colors=['green', 'yellow', 'orange', 'red'],
        vmin=0,
        vmax=max_popularity,
        caption='Station Popularity'
    )
    
    # Add white background to colormap
    colormap.add_to(m)
    
    # Add station markers
    for _, station in stations_df.iterrows():
        # Create popup with station information
        popup_html = f"""
        <strong>{station['name']}</strong><br>
        <strong>Popularity:</strong> {station['popularity']}<br>
        <strong>Location:</strong> {station['lat']:.4f}, {station['lng']:.4f}
        """
        
        # Determine marker color based on popularity
        color = colormap(station['popularity'])
        
        # Add marker to all stations group
        folium.CircleMarker(
            location=[station['lat'], station['lng']],
            radius=5,
            color='black',
            fill=True,
            fill_color=color,
            fill_opacity=0.7,
            popup=folium.Popup(popup_html, max_width=300)
        ).add_to(stations_group)
        
        # Add popular stations to a separate group
        if station['popularity'] >= popularity_threshold:
            folium.CircleMarker(
                location=[station['lat'], station['lng']],
                radius=8,
                color='black',
                fill=True,
                fill_color='red',
                fill_opacity=0.9,
                popup=folium.Popup(popup_html, max_width=300)
            ).add_to(popular_group)
        
        # Add to cluster group
        folium.Marker(
            location=[station['lat'], station['lng']],
            popup=folium.Popup(popup_html, max_width=300),
        ).add_to(clusters_group)
    
    # Add all layers to map
    stations_group.add_to(m)
    popular_group.add_to(m)
    clusters_group.add_to(m)
    
    # Add layer control
    folium.LayerControl().add_to(m)
    
    # Save the map to a file
    os.makedirs(output_dir, exist_ok=True)
    map_filename = os.path.join(output_dir, "interactive_station_map.html")
    m.save(map_filename)
    print(f"Interactive station map saved to {map_filename}")
    
    return m

def calculate_od_matrix(trips_df, station_mapping):
    """Calculate the origin-destination matrix from trips"""
    # Filter trips to only include those between mapped stations
    mapped_trips = trips_df[
        trips_df['start_station_name'].isin(station_mapping.keys()) & 
        trips_df['end_station_name'].isin(station_mapping.keys())
    ]
    
    # Initialize OD matrix with zeros
    num_stations = len(station_mapping)
    od_matrix = pd.DataFrame(
        np.zeros((num_stations, num_stations)),
        index=range(num_stations),
        columns=range(num_stations)
    )
    
    # Fill the OD matrix with trip counts
    for _, trip in mapped_trips.iterrows():
        origin_idx = station_mapping[trip['start_station_name']]
        dest_idx = station_mapping[trip['end_station_name']]
        od_matrix.iloc[origin_idx, dest_idx] += 1
    
    return od_matrix

def create_flow_map(od_matrix, stations_df, title="Bike Flow Map", flow_scale=1.0, min_trips=5):
    """Create an interactive flow map showing trip patterns between stations with improved visual hierarchy"""
    # Calculate map center
    center_lat = stations_df['lat'].mean()
    center_lng = stations_df['lng'].mean()
    
    # Create base map
    m = folium.Map(
        location=[center_lat, center_lng],
        zoom_start=13,
        tiles=None,
        control_scale=True
    )
    
    # Add multiple tile layers with proper attribution
    folium.TileLayer('cartodbpositron', name='CartoDB Positron', control=True).add_to(m)
    folium.TileLayer('openstreetmap', name='OpenStreetMap', control=True).add_to(m)
    folium.TileLayer('cartodbdark_matter', name='CartoDB Dark Matter', control=True).add_to(m)
    
    # Add Esri tile layers with attribution
    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
        attr="Esri",
        name="Esri Satellite",
        control=True
    ).add_to(m)
    
    # Add map title
    title_html = f'''
        <h3 align="center" style="font-size:16px"><b>{title}</b></h3>
    '''
    m.get_root().html.add_child(folium.Element(title_html))
    
    # Create feature groups for different flow categories
    major_flows = folium.FeatureGroup(name="Major Flows (Top 20%)", show=True)
    medium_flows = folium.FeatureGroup(name="Medium Flows", show=True)
    minor_flows = folium.FeatureGroup(name="Minor Flows", show=False)
    
    # Find the maximum flow value for normalization
    max_flow = od_matrix.max().max()
    
    # Calculate flow thresholds based on percentiles
    all_flows = od_matrix.values.flatten()
    all_flows = all_flows[all_flows >= min_trips]
    
    if len(all_flows) > 0:
        top_threshold = np.percentile(all_flows, 80)
        medium_threshold = np.percentile(all_flows, 50)
    else:
        top_threshold = max_flow
        medium_threshold = max_flow / 2
    
    # Create colormap for flow lines with improved legend styling
    colormap = branca.colormap.LinearColormap(
        colors=['blue', 'green', 'yellow', 'orange', 'red'],
        vmin=min_trips,
        vmax=max_flow,
        caption='Trip Volume'
    )
    
    # Add colormap to map with white background and border
    colormap = colormap.add_to(m)
    
    # Add CSS to style the colormap legend with white background
    legend_style = """
    <style>
        .legend {
            background-color: rgba(255, 255, 255, 0.9) !important;
            border-radius: 5px !important;
            padding: 6px !important;
            box-shadow: 0 0 5px rgba(0, 0, 0, 0.2) !important;
        }
    </style>
    """
    m.get_root().html.add_child(folium.Element(legend_style))
    
    # Generate flow colors for each station (to organize flows by origin)
    station_colors = {}
    for i, station in enumerate(stations_df['name']):
        hue = (i * 30) % 360  # Distribute colors around the color wheel
        station_colors[station] = hue
    
    # Add flow lines between stations
    flow_count = 0
    for i in range(len(od_matrix)):
        for j in range(len(od_matrix)):
            if i != j and od_matrix.iloc[i, j] >= min_trips:
                # Get station coordinates
                start_station = stations_df.iloc[i]
                end_station = stations_df.iloc[j]
                
                # Get flow volume
                flow = od_matrix.iloc[i, j]
                
                # Determine relative importance of flow
                rel_flow = flow / max_flow
                
                # Calculate width scaling for better visual hierarchy 
                # REDUCED MAXIMUM THICKNESS for major flows
                width = 1 + (rel_flow)**1.2 * flow_scale * 8  # Reduced max width from 15 to 8
                
                # Calculate opacity based on flow volume
                opacity = min(0.85, 0.2 + rel_flow * 0.65)
                
                # Create curved line with adaptive curvature
                curve_strength = min(0.3, 0.05 + 0.15 * rel_flow)
                
                # Add small positional offsets to prevent exact overlaps
                angle_offset = (flow_count % 8) * (math.pi/4)
                offset_distance = 0.0001
                dx = math.cos(angle_offset) * offset_distance
                dy = math.sin(angle_offset) * offset_distance
                
                # Create curve with adjusted endpoints
                points = create_bezier_curve_with_strength(
                    start_station['lng'] + dx, start_station['lat'] + dy,
                    end_station['lng'] + dx, end_station['lat'] + dy,
                    curve_strength
                )
                
                # Create popup with flow information
                popup_html = f"""
                <strong>From:</strong> {start_station['name']}<br>
                <strong>To:</strong> {end_station['name']}<br>
                <strong>Trips:</strong> {flow}
                """
                
                # Determine flow category and target layer
                if flow >= top_threshold:
                    # Major flow - use standard color scheme for visibility
                    color = colormap(flow)
                    target_group = major_flows
                    # Keep major flows visible but not too thick
                    width = min(width, 6.5)  # Cap maximum width
                elif flow >= medium_threshold:
                    # Medium flow - use color gradients by origin station
                    base_hue = station_colors.get(start_station['name'], 0)
                    saturation = 80 + rel_flow * 20
                    lightness = 50
                    color = f"hsl({base_hue}, {saturation}%, {lightness}%)"
                    target_group = medium_flows
                else:
                    # Minor flow - more transparent with origin-based coloring
                    base_hue = station_colors.get(start_station['name'], 0)
                    color = f"hsl({base_hue}, 70%, 60%)"
                    target_group = minor_flows
                    # Make minor flows more subtle
                    opacity = max(0.1, opacity - 0.1)
                
                # Add the flow line to the appropriate group
                folium.PolyLine(
                    points,
                    color=color,
                    weight=width,
                    opacity=opacity,
                    popup=folium.Popup(popup_html, max_width=300),
                    tooltip=f"{start_station['name']} → {end_station['name']}: {flow} trips"
                ).add_to(target_group)
                
                flow_count += 1
    
    # Add station markers
    stations_group = folium.FeatureGroup(name="Stations", show=True)
    
    for _, station in stations_df.iterrows():
        folium.CircleMarker(
            location=[station['lat'], station['lng']],
            radius=5,
            color='black',
            fill=True,
            fill_color='blue',
            fill_opacity=0.7,
            popup=folium.Popup(f"<strong>{station['name']}</strong>", max_width=300)
        ).add_to(stations_group)
    
    # Add all layers to map
    major_flows.add_to(m)
    medium_flows.add_to(m)
    minor_flows.add_to(m)
    stations_group.add_to(m)
    
    # Add layer control
    folium.LayerControl().add_to(m)
    
    return m

def create_time_period_flow_maps(trips_df, stations_df, station_mapping, output_dir="results"):
    """Create separate flow maps for each time period"""
    # Create directory for time period maps if it doesn't exist
    time_periods_dir = os.path.join(output_dir, "time_periods")
    os.makedirs(time_periods_dir, exist_ok=True)
    
    print("Generating flow maps and OD matrix visualizations...")
    
    # Process each time period separately
    for period_name, (start_hour, end_hour) in TIME_PERIODS.items():
        # Filter trips for this time period
        period_trips = trips_df[(trips_df['start_hour'] >= start_hour) & 
                              (trips_df['start_hour'] < end_hour)]
        
        print(f"Processing {period_name} with {len(period_trips)} trips")
        
        # Skip if no trips in this period
        if len(period_trips) == 0:
            print(f"No trips found for {period_name}, skipping...")
            continue
            
        # Calculate OD matrix for this time period
        od_matrix = calculate_od_matrix(period_trips, station_mapping)
        
        # Create flow map for this time period
        flow_map = create_flow_map(
            od_matrix, 
            stations_df,
            title=f"Bike Flow Map - {period_name}",
            flow_scale=3.0,  # Reduced flow scale from 5.0 to 3.0 for thinner lines
            min_trips=5
        )
        
        # Save the map with a unique filename for each period
        safe_period_name = period_name.replace(" ", "_").replace("(", "").replace(")", "").replace("-", "to")
        map_filename = f"flow_map_{safe_period_name}.html"
        flow_map.save(os.path.join(time_periods_dir, map_filename))
        print(f"Flow map for {period_name} saved to {os.path.join(time_periods_dir, map_filename)}")
    
    # Create an overall flow map with all trips
    od_matrix_all = calculate_od_matrix(trips_df, station_mapping)
    all_day_map = create_flow_map(
        od_matrix_all, 
        stations_df,
        title="Bike Flow Map - All Day",
        flow_scale=3.0,  # Reduced flow scale for thinner lines
        min_trips=5
    )
    
    all_day_map.save(os.path.join(output_dir, "flow_map.html"))
    print(f"Flow map saved to {os.path.join(output_dir, 'flow_map.html')}")
    
    print("Flow visualizations completed")

def main():
    """Main function to run the analysis pipeline"""
    # Parse runtime configuration
    data_dir = RUNTIME_CONFIG.get('DATA_DIR', 'data/citibike')
    output_dir = RUNTIME_CONFIG.get('OUTPUT_DIR', 'results')
    max_stations = RUNTIME_CONFIG.get('MAX_STATIONS', 50)
    sample_rate = RUNTIME_CONFIG.get('SAMPLE_RATE', 1.0)
    data_files_pattern = RUNTIME_CONFIG.get('DATA_FILES_PATTERN', 'JC2024*citibiketripdata.csv')
    
    print("Starting analysis with parameters:")
    print({
        'data_dir': data_dir,
        'output_dir': output_dir,
        'max_stations': max_stations,
        'sample_rate': sample_rate
    })
    
    # Load and clean data
    trips_df = load_citibike_data(data_dir, data_files_pattern, sample_rate)
    
    # Get top stations and create mapping
    station_mapping, station_data, top_stations, stations_df = get_top_stations(trips_df, max_stations)
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    print("\nGenerating visualizations...")
    
    # Create interactive station map
    interactive_station_map = create_interactive_station_map(
        stations_df,
        map_title="Citibike Stations",
        output_dir=output_dir
    )
    
    # Create flow maps for different time periods
    create_time_period_flow_maps(trips_df, stations_df, station_mapping, output_dir)
    
    print("\nAnalysis completed successfully!")

if __name__ == "__main__":
    main()

Starting analysis with parameters:
{'data_dir': 'data/citibike', 'output_dir': 'results', 'max_stations': 50, 'sample_rate': 1}
Loading Citibike data with 100.0% sampling rate
Found 12 files to process
Processing file: JC202401citibiketripdata.csv
Extracted month: 1
Loaded 50661 sampled trips from JC202401citibiketripdata.csv
Processing file: JC202402citibiketripdata.csv
Extracted month: 2
Loaded 55613 sampled trips from JC202402citibiketripdata.csv
Processing file: JC202403citibiketripdata.csv
Extracted month: 3
Loaded 65581 sampled trips from JC202403citibiketripdata.csv
Processing file: JC202404citibiketripdata.csv
Extracted month: 4
Loaded 79116 sampled trips from JC202404citibiketripdata.csv
Processing file: JC202405citibiketripdata.csv
Extracted month: 5
Loaded 97479 sampled trips from JC202405citibiketripdata.csv
Processing file: JC202406citibiketripdata.csv
Extracted month: 6
Loaded 111115 sampled trips from JC202406citibiketripdata.csv
Processing file: JC202407citibiketripdata